# Experiment 3: Embedding Vectorizers vs TF-IDF with MLflow

Comparing TF-IDF against DistilBERT contextual Transformer embeddings for Logistic Regression sentiment classification, logging all metrics and models to MLflow & DagsHub.

In [ ]:
import os
import re
import string
import warnings
import pandas as pd
import numpy as np
import scipy.sparse
import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import mlflow
import mlflow.sklearn
import dagshub

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

## 1. Setup MLflow Tracking

In [ ]:
DATA_PATH = '../data/balanced_sentiment_dataset.csv' if os.path.exists('../data/balanced_sentiment_dataset.csv') else 'data/balanced_sentiment_dataset.csv'

CONFIG = {
    "data_path": DATA_PATH,
    "test_size": 0.2,
    "mlflow_tracking_uri": os.getenv("MLFLOW_TRACKING_URI", "https://dagshub.com/rohitkr8527/sentiment-analysis.mlflow"),
    "dagshub_repo_owner": os.getenv("DAGSHUB_REPO_OWNER", "rohitkr8527"),
    "dagshub_repo_name": os.getenv("DAGSHUB_REPO_NAME", "sentiment-analysis"),
    "experiment_name": "Embedding Vectorizers with Logistic Regression"
}

mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
dagshub_token = os.getenv("DAGSHUB_TOKEN") or os.getenv("sentiment_analysis", "")
if dagshub_token:
    os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_token
    os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token
    dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)

mlflow.set_experiment(CONFIG["experiment_name"])
print(f"MLflow Experiment '{CONFIG['experiment_name']}' ready.")

## 2. Load and Preprocess Data

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = ''.join([char for char in text if not char.isdigit()])
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return " ".join(words)

df = pd.read_csv(CONFIG["data_path"])
df['text'] = df['text'].apply(preprocess_text)
print(f"Loaded dataset with {len(df)} samples.")

## 3. Define DistilBERT Embedding Extractor

In [ ]:
print("Loading DistilBERT tokenizer and transformer model...")
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
transformer_model = AutoModel.from_pretrained('distilbert-base-uncased')
transformer_model.eval()

def extract_transformer_embeddings(texts, batch_size=32):
    vectors = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
            outputs = transformer_model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            vectors.append(cls_embeddings)
    return np.vstack(vectors)

## 4. Run Experiment with MLflow Logging

In [ ]:
sample_df = df.sample(n=min(len(df), 5000), random_state=42)
X_texts = sample_df['text'].tolist()
y = sample_df['sentiment'].values

with mlflow.start_run(run_name="Embeddings vs TF-IDF Comparison") as parent_run:
    # --- 1. TF-IDF Trial ---
    with mlflow.start_run(run_name="TF-IDF with LogisticRegression", nested=True):
        tfidf_vec = TfidfVectorizer(max_features=5000)
        X_tfidf = tfidf_vec.fit_transform(X_texts)
        X_tr, X_te, y_tr, y_te = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

        mlflow.log_params({"vectorizer": "TF-IDF", "max_features": 5000, "C": 1.0, "max_iter": 1000})
        clf_tfidf = LogisticRegression(max_iter=1000, random_state=42)
        clf_tfidf.fit(X_tr, y_tr)

        y_pred = clf_tfidf.predict(X_te)
        metrics_tfidf = {
            "accuracy": float(accuracy_score(y_te, y_pred)),
            "precision": float(precision_score(y_te, y_pred)),
            "recall": float(recall_score(y_te, y_pred)),
            "f1_score": float(f1_score(y_te, y_pred))
        }
        mlflow.log_metrics(metrics_tfidf)
        mlflow.sklearn.log_model(clf_tfidf, "model")
        print("TF-IDF Metrics:", metrics_tfidf)

    # --- 2. Transformer Embeddings Trial ---
    with mlflow.start_run(run_name="DistilBERT with LogisticRegression", nested=True):
        X_trans = extract_transformer_embeddings(X_texts)
        X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(X_trans, y, test_size=0.2, random_state=42)

        mlflow.log_params({"vectorizer": "DistilBERT", "embedding_dim": 768, "C": 1.0, "max_iter": 1000})
        clf_trans = LogisticRegression(max_iter=1000, random_state=42)
        clf_trans.fit(X_tr_t, y_tr_t)

        y_pred_t = clf_trans.predict(X_te_t)
        metrics_trans = {
            "accuracy": float(accuracy_score(y_te_t, y_pred_t)),
            "precision": float(precision_score(y_te_t, y_pred_t)),
            "recall": float(recall_score(y_te_t, y_pred_t)),
            "f1_score": float(f1_score(y_te_t, y_pred_t))
        }
        mlflow.log_metrics(metrics_trans)
        mlflow.sklearn.log_model(clf_trans, "model")
        print("DistilBERT Metrics:", metrics_trans)